# Optional Phase 3: activation steering
Fit answer-letter vectors on training questions and evaluate separate held-out questions. This can be much slower than Phase 2: the default includes four targets, four nonzero strengths, and vector/random controls. A changed answer does not establish deliberate concealment or introspective awareness.

## Setup
Select **Runtime → Change runtime type → T4 GPU**. Upload the project ZIP containing a top-level `hint-faithfulness/` folder. You may select a saved results ZIP at the same time; the notebook restores it after the project, so you do not need to repack archives. Include local Phase 1 records when continuing an experiment. Do not include credentials or virtual environments. Free GPU availability is not guaranteed.

In [ ]:
from google.colab import files
from pathlib import Path
import io, os, zipfile

# Select the project ZIP and any saved results ZIP together.
uploaded = files.upload()
archives = [name for name in uploaded if name.endswith(".zip")]
if not archives:
    raise ValueError("Upload the project ZIP, optionally with a saved results ZIP")
root = Path("/content").resolve()
def contains_project(name):
    with zipfile.ZipFile(io.BytesIO(uploaded[name])) as archive:
        return "hint-faithfulness/src/run.py" in archive.namelist()
# Install project files first; restore saved data and human labels afterward.
archives.sort(key=lambda name: 0 if contains_project(name) else 1)
for name in archives:
    with zipfile.ZipFile(io.BytesIO(uploaded[name])) as archive:
        for member in archive.infolist():
            destination = (root / member.filename).resolve()
            if not destination.is_relative_to(root):
                raise ValueError("Archive path escapes /content")
        archive.extractall(root)
project = root / "hint-faithfulness"
if not (project / "src" / "run.py").exists():
    raise ValueError("Project ZIP must contain hint-faithfulness/src/run.py")
os.chdir(project)


In [ ]:
!python -m pip install -q -r requirements.txt
import torch
assert torch.cuda.is_available(), "Select a GPU runtime before loading weights"
print(torch.cuda.get_device_name(0))
!python -m unittest discover -s tests -q


## Predict before running
Write your own prediction below. This is a dated expectation, not a result. Keep it when the outcome disagrees.

In [ ]:
from datetime import datetime, timezone
PREDICTION = ""  # Fill in your prediction before running the experiment.
assert PREDICTION.strip(), "Write a prediction first"
with Path("LOG.md").open("a") as log:
    log.write(f"\n\n{datetime.now(timezone.utc).isoformat()} — prediction\n{PREDICTION}\n")


## Pilot the intervention
The script changes the last prompt position (which predicts the first generated token) and subsequent generated positions. All conditions use matched generation seeds. Each target gets a random direction with the same norm. Vectors are built from matched hinted/unhinted prompts; every target letter needs training examples.

In [ ]:
RECORDS = "data/runs/small.jsonl"
assert Path(RECORDS).exists()
!python -m src.phase3 --records {RECORDS} --out-dir results/phase3-pilot --limit 5 --max-new-tokens 4096 --device cuda


In [ ]:
from IPython.display import display, Image
display(Image("results/phase3-pilot/steering_rates.png"))


## Inspect before expanding
First inspect whether steering moves answers toward the intended letter beyond the random-direction control, and check incoherence and failure counts. If the effect is absent, stop and debug the intervention. To expand, run the following command in a fresh output directory. This may exceed a single free GPU session; use `--resume` with unchanged arguments after restoring saved artifacts.

In [ ]:
# Run after inspecting the pilot; uncomment to launch the full held-out run.
# !python -m src.phase3 --records {RECORDS} --out-dir results/phase3 --max-new-tokens 4096 --device cuda


## Label explanations yourself
Read `results/phase3-pilot/LABELING.md`, then fill the blank `human_label` column in `explanation_labels.csv`. Keep random-direction controls in the review. Allowed labels are `outside_nudge`, `ordinary_argument`, and `incoherent`; record uncertainty in notes. The script does not invent labels. After editing, summarize without reloading weights:

In [ ]:
!python -m src.phase3 --summarize-only --out-dir results/phase3-pilot
print(Path("results/phase3-pilot/EXPLANATION_SUMMARY.md").read_text())


## Save your work
Download results before the Colab session ends. The archive includes data, logs, and experiment outputs, but excludes downloaded weights and credentials.

In [ ]:
from google.colab import files
with zipfile.ZipFile("research-results.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for folder in (Path("data"), Path("results")):
        if folder.exists():
            for item in folder.rglob("*"):
                if item.is_file():
                    archive.write(item, Path("hint-faithfulness") / item)
    if Path("LOG.md").exists():
        archive.write("LOG.md", "hint-faithfulness/LOG.md")
files.download("research-results.zip")
